### corpus 생성

In [23]:
sentences = [
"i want chinese food",
"i want thai food",
"i want italian food",
"i want a restaurant",
"i want to eat",
"i want to eat chinese food",
"i want to eat thai food",
"i am looking for chinese food",
"i am looking for a restaurant",
"can you find a chinese restaurant",
"can you find a thai restaurant",
"can you find an italian restaurant",
"tell me about chinese restaurants",
"tell me about thai restaurants",
"where can i eat chinese food",
]

# 문장들을 토큰 단위로 분해 후 corpus에 저장
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

corpus = [tokenize(s) for s in sentences]

### corpus 확인(출력)

In [24]:
# corpus 안의 값들 출력
for sentence in corpus[:3]:
    print(sentence)

['<s>', 'i', 'want', 'chinese', 'food', '</s>']
['<s>', 'i', 'want', 'thai', 'food', '</s>']
['<s>', 'i', 'want', 'italian', 'food', '</s>']


### 단어의 빈도 수 저장

In [25]:
from collections import Counter

unigram_counts = Counter()

# corpus 안의 단어들의 누적 값을 unigram_counts에 저장
for sentence in corpus:
    unigram_counts.update(sentence)

print(unigram_counts.most_common(10))

[('<s>', 15), ('</s>', 15), ('i', 10), ('want', 7), ('food', 7), ('chinese', 6), ('restaurant', 5), ('thai', 4), ('a', 4), ('eat', 4)]


### Unigram Counts, Probability

In [26]:
# 총 단어의 개수를 total_words에 저장
total_words = sum(unigram_counts.values())

# 각 단어의 등장 확률을 계산하여 출력하는 함수(단어 개수 / 총 단어 수)
def unigram_prob(word):
    return unigram_counts[word]/total_words

print(unigram_prob("want"))
print(unigram_prob("food"))

0.06481481481481481
0.06481481481481481


### Bigram Counts

In [27]:
bigram_counts = Counter()

for sentence in corpus:
    # 맨 앞의 인덱스를 고려해 -1 처리
    # 단어의 쌍과 해당 쌍이 나오는 횟수를 bigram_counts에 저장
    for i in range(len(sentence) - 1):
        bigram = (sentence[i], sentence[i + 1])
        bigram_counts[bigram] += 1

# 가장 많이 등장하는 5개의 쌍을 그 횟수와 함께 출력
for bigram, count in bigram_counts.most_common(5):
    print(bigram, count)

('<s>', 'i') 9
('i', 'want') 7
('food', '</s>') 7
('restaurant', '</s>') 5
('chinese', 'food') 4


### Bigram Probability

In [ ]:
# 두 단어를 받아 (두 단어가 연속으로 등장할 확률 / 이전 단어가 단독으로 등장할 확를)을 반환
def bigram_prob(previous_word, word):
    numerator = bigram_counts[(previous_word), word]
    denominator = unigram_counts[previous_word]

    # divide by zero 예외 처리
    if denominator == 0:
        return 0

    return numerator / denominator

print("P(want | i) =", bigram_prob("i", "want"))
print("P(food | chinese) =", bigram_prob("chinese", "food"))
print("P(restaurant | a) =", bigram_prob("a", "restaurant"))

P(want | i) = 0.7
P(food | chinese) = 0.6666666666666666
P(restaurant | a) = 0.5


### Next Word Prediction (Bigram)

In [29]:
# 특정 단어를 입력했을 때, 다음에 올 가능성이 높은 단어를 내림차순으로 3개 출력
def predict_bigram(previous_word, top_k = 3):
    candidates = []

    for (w1, w2), count in bigram_counts.items():
        if w1 == previous_word:
            probability = bigram_prob(w1, w2)
            candidates.append((w2, probability))

    return sorted(
        candidates,
        key = lambda x:x[1],
        reverse=True
    )[:top_k]

predict_bigram("want")

[('to', 0.42857142857142855),
 ('chinese', 0.14285714285714285),
 ('thai', 0.14285714285714285)]

### Generate Restaurant Sentences

In [54]:
import random

# 단어를 하나 받아, 다음으로 올 단어 하나를 반환
def sample_next_word(previous_word):
    candidates = []
    weights = []

    # 해당 단어의 다음으로 올 수 있는 단어들을 candidate에 삽입
    # weight = 해당 단어의 바로 다음으로 현재 counts의 단어가 온 횟수
    for (w1, w2), count in bigram_counts.items():
        if w1 == previous_word:
            candidates.append(w2)
            weights.append(count)

    # 다음 단어로 오는 단어가 존재하지 않을 경우 문장을 끝마침
    if not candidates:
        return "</s>"
    return random.choices(
        candidates,
        weights=weights,
        k = 1
    )[0]

# 학습한 데이터를 바탕으로 문장을 생성
# max_length: 문장의 최대 길이
def generate_sentence(max_length = 15):
    current = "<s>"
    output = []

    for _ in range(max_length):
        next_word = sample_next_word(current)
        if next_word == "</s>":
            break
        output.append(next_word)
        current = next_word
    return " ".join(output)

for _ in range(10):
    print(generate_sentence())

where can you find a thai food
can you find a thai restaurants
i want a restaurant
where can you find a restaurant
tell me about thai food
i want to eat
i am looking for chinese restaurant
tell me about chinese food
where can you find a thai restaurant
i want to eat chinese food


### Perplexity Metric

In [31]:
import math

# 각 문장의 예측 가능한 정도를 log probability로 반환
def perplexity(sentence):
    tokens = tokenize(sentence)
    log_prob = 0
    n = len(tokens) - 1

    for i in range(n):
        p = bigram_prob(
            tokens[i],
            tokens[i + 1]
        )
        if p == 0:
            return float("inf")
        log_prob += math.log(p)
    return math.exp(-log_prob / n)

test_sentences = [
"i want chinese food",
"i want thai food",
"restaurant chinese want i",
]

for s in test_sentences:
    print(s, perplexity(s))

i want chinese food 1.9036539387158786
i want thai food 2.016395636994333
restaurant chinese want i inf


### Trigram Probability

In [ ]:
trigram_counts = Counter()

# 모든 sentence를 순회하며 연속하는 세 단어의 등장 횟수를 trigram_counts에 저장
for sentence in corpus:
    for i in range(len(sentence) - 2):
        w1, w2, w3 = sentence[i:i + 3]
        trigram_counts[(w1, w2, w3)] += 1

# trigram_counts와 bigram_counts를 바탕으로 세 단어가 연속으로 올 확률을 출력
def trigram_prob(w1, w2, w3):
    numerator = trigram_counts[(w1, w2, w3)]
    denominator = bigram_counts[(w1, w2)]

    if denominator == 0:
        print("zero")
        return 0
    return numerator / denominator

print(trigram_prob("i", "want", "chinese"))

0.14285714285714285


### Next Word Prediction (Trigram)

In [51]:
# 특정 단어 쌍을 입력했을 때, 다음에 올 가능성이 높은 단어를 내림차순으로 3개 출력
def predict_trigram(previous_word1, previous_word2, top_k = 3):
    candidates = []

    for (w1, w2, w3), count in trigram_counts.items():
        if w1 == previous_word1 and w2 == previous_word2:
            probability = trigram_prob(w1, w2, w3)
            candidates.append((w3, probability))

    return sorted(
        candidates,
        key = lambda x:x[1],
        reverse=True
    )[:top_k]

predict_trigram("i", "want")
            

[('to', 0.42857142857142855),
 ('chinese', 0.14285714285714285),
 ('thai', 0.14285714285714285)]

### Generate Restaurant Sentences

In [55]:
def sample_next_word_tri(previous_word1, previous_word2):
    candidates = []
    weights = []

    for (w1, w2, w3), count in trigram_counts.items():
        if w1 == previous_word1 and w2 == previous_word2:
            candidates.append(w3)
            weights.append(count)
    if not candidates:
        return "</s>"

    return random.choices(
        candidates,
        weights=weights,
        k = 1
    )[0]


def generate_sentence_tri(max_length=15):
    current = "<s>"
    current2 = sample_next_word(current)
    output = []

    for _ in range(max_length):
        next_word = sample_next_word_tri(current, current2)
        if next_word == "</s>":
            break
        output.append(next_word)
        current = current2
        current2 = next_word
    return " ".join(output)

for _ in range(5):
    print(generate_sentence_tri())

you find an italian restaurant
me about thai restaurants
want to eat chinese food
me about thai restaurants
am looking for chinese food


### Perplexity Metric (Unigram, Bigram, Trigram)

In [66]:
def perplexity_unigram(sentence):
    tokens = tokenize(sentence)
    log_prob = 0
    n = len(tokens)
    for i in range(n):
        p = unigram_prob(
            tokens[i]
        )
        if p == 0:
            return float("inf")
        log_prob += math.log(p)
    return math.exp(-log_prob / n)

def perplexity_trigram(sentence):
    tokens = tokenize(sentence)
    log_prob = 0
    n = len(tokens) - 2
    for i in range(n):
        p = trigram_prob(
            tokens[i],
            tokens[i + 1],
            tokens[i + 2]
        )
        if p == 0:
            return float("inf")
        log_prob += math.log(p)
    return math.exp(-log_prob / n)
test_sentences_task = [
    "i want italian restaurants",
    "where can you find thai food",
    "i am looking for italian food",
    "can you tell me about chinese food",
    "where can i find chinese food",
]

print("=====UNIGRAM=====")
for s in test_sentences_task:
    print(s, perplexity_unigram(s))
print()

print("=====BIGRAM=====")
for s in test_sentences_task:
    print(s, perplexity(s))
print()

print("=====TRIGRAM=====")
for s in test_sentences_task:
    print(s, perplexity_trigram(s))



=====UNIGRAM=====
i want italian restaurants 17.121178847001605
where can you find thai food 23.11896429264312
i am looking for italian food 22.816482268728738
can you tell me about chinese food 23.521052477922876
where can i find chinese food 18.905889867431473

=====BIGRAM=====
i want italian restaurants inf
where can you find thai food inf
i am looking for italian food inf
can you tell me about chinese food inf
where can i find chinese food inf

=====TRIGRAM=====
i want italian restaurants inf
where can you find thai food inf
i am looking for italian food inf
can you tell me about chinese food inf
where can i find chinese food inf
